# Notebook 2 — Parent RAG
**Livro:** Os Sertões — Euclides da Cunha  
**Estratégia:** Chunks filhos pequenos para busca + chunks pais grandes para contexto de geração

## 1. Instalação de dependências

In [ ]:
!pip install -q anthropic pypdf chromadb sentence-transformers langchain langchain-community tiktoken

## 2. Importações e configuração

In [ ]:
import os
import re
import json
import anthropic
import chromadb
import urllib.request
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from typing import List, Dict

client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))
print("Cliente Anthropic inicializado.")

## 3. Download e extração do PDF

In [ ]:
PDF_URL = "https://fundar.org.br/wp-content/uploads/2021/06/os-sertoes.pdf"
PDF_PATH = "os-sertoes.pdf"

if not os.path.exists(PDF_PATH):
    print("Baixando o PDF...")
    urllib.request.urlretrieve(PDF_URL, PDF_PATH)

reader = PdfReader(PDF_PATH)
full_text = "".join(page.extract_text() or "" for page in reader.pages)
clean_text = re.sub(r'\s+', ' ', full_text).strip()
print(f"Texto extraído: {len(clean_text):,} caracteres")

## 4. Criação de chunks Pai e Filho (Parent-Child chunking)

> **Ideia central do Parent RAG:**  
> - **Chunks filhos** (pequenos, ~300 chars) → usados para busca semântica precisa  
> - **Chunks pais** (grandes, ~1500 chars) → usados como contexto rico para geração  
> Cada filho conhece seu pai via `parent_id`.

In [ ]:
def create_parent_chunks(text: str, parent_size: int = 1500, overlap: int = 200) -> List[Dict]:
    """Cria chunks pais grandes com sobreposição."""
    parents = []
    start = 0
    idx = 0
    while start < len(text):
        end = start + parent_size
        chunk = text[start:end].strip()
        if chunk:
            parents.append({"id": f"parent_{idx}", "text": chunk})
            idx += 1
        start += parent_size - overlap
    return parents

def create_child_chunks(parent: Dict, child_size: int = 300, overlap: int = 50) -> List[Dict]:
    """Divide um chunk pai em vários filhos menores."""
    text = parent["text"]
    children = []
    start = 0
    idx = 0
    while start < len(text):
        end = start + child_size
        chunk = text[start:end].strip()
        if chunk and len(chunk) > 50:  # ignora fragmentos muito pequenos
            children.append({
                "id": f"{parent['id']}_child_{idx}",
                "text": chunk,
                "parent_id": parent["id"]
            })
            idx += 1
        start += child_size - overlap
    return children

# Cria estrutura hierárquica
parent_chunks = create_parent_chunks(clean_text, parent_size=1500, overlap=200)
child_chunks = []
for parent in parent_chunks:
    child_chunks.extend(create_child_chunks(parent, child_size=300, overlap=50))

# Mapa de id → texto do pai para acesso rápido
parent_map = {p["id"]: p["text"] for p in parent_chunks}

print(f"Chunks pais  : {len(parent_chunks)}")
print(f"Chunks filhos: {len(child_chunks)}")
print(f"\nExemplo — pai 0:\n{parent_chunks[0]['text'][:300]}...")
print(f"\nExemplo — filho 0 (pai=parent_0):\n{child_chunks[0]['text']}")

## 5. Indexação dos chunks filhos no ChromaDB

In [ ]:
embed_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

chroma_client = chromadb.Client()
child_collection = chroma_client.get_or_create_collection(
    name="parent_rag_children",
    metadata={"hnsw:space": "cosine"}
)

BATCH_SIZE = 100
for i in range(0, len(child_chunks), BATCH_SIZE):
    batch = child_chunks[i:i + BATCH_SIZE]
    texts = [c["text"] for c in batch]
    ids   = [c["id"]   for c in batch]
    metas = [{"parent_id": c["parent_id"]} for c in batch]
    embeddings = embed_model.encode(texts, show_progress_bar=False).tolist()
    child_collection.add(documents=texts, embeddings=embeddings, ids=ids, metadatas=metas)
    if i % 500 == 0:
        print(f"  Indexados {i + len(batch)}/{len(child_chunks)} filhos...")

print(f"\nIndexação concluída: {child_collection.count()} filhos no banco.")

## 6. Pipeline Parent RAG

In [ ]:
def retrieve_parents(query: str, k_children: int = 10) -> List[str]:
    """
    1. Busca os k filhos mais similares à query.
    2. Mapeia cada filho ao seu pai (sem duplicatas).
    3. Retorna os textos dos pais únicos.
    """
    query_emb = embed_model.encode([query]).tolist()
    results = child_collection.query(
        query_embeddings=query_emb,
        n_results=k_children,
        include=["metadatas"]
    )
    seen_parents = set()
    parent_texts = []
    for meta in results["metadatas"][0]:
        pid = meta["parent_id"]
        if pid not in seen_parents:
            seen_parents.add(pid)
            parent_texts.append(parent_map[pid])
    return parent_texts

def generate_answer(query: str, context_chunks: List[str]) -> str:
    context = "\n\n---\n\n".join(context_chunks)
    prompt = f"""Você é um assistente especializado em literatura brasileira.
Use APENAS o contexto abaixo para responder à pergunta com detalhes.
Se a informação não estiver no contexto, diga que não encontrou.

CONTEXTO:
{context}

PERGUNTA: {query}

RESPOSTA:"""
    message = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=1024,
        messages=[{"role": "user", "content": prompt}]
    )
    return message.content[0].text

def parent_rag(query: str, k_children: int = 10) -> dict:
    parents = retrieve_parents(query, k_children=k_children)
    answer = generate_answer(query, parents)
    return {"query": query, "n_parents_used": len(parents), "answer": answer}

print("Pipeline Parent RAG pronto!")

## 7. Respondendo às 5 questões

In [ ]:
questions = [
    "Qual é a visão de Euclides da Cunha sobre o ambiente natural do sertão nordestino e como ele influencia a vida dos habitantes?",
    "Quais são as principais características da população sertaneja descritas por Euclides da Cunha? Como ele relaciona essas características com o ambiente em que vivem?",
    "Qual foi o contexto histórico e político que levou à Guerra de Canudos, segundo Euclides da Cunha?",
    "Como Euclides da Cunha descreve a figura de Antônio Conselheiro e seu papel na Guerra de Canudos?",
    "Quais são os principais aspectos da crítica social e política presentes em Os Sertões? Como esses aspectos refletem a visão do autor sobre o Brasil da época?"
]

results = []
for i, q in enumerate(questions, 1):
    print(f"\n{'='*70}")
    print(f"QUESTÃO {i}: {q}")
    print('='*70)
    result = parent_rag(q, k_children=10)
    results.append(result)
    print(f"[Pais utilizados: {result['n_parents_used']}]")
    print(result["answer"])

## 8. Salvando resultados

In [ ]:
with open("resultados_parent_rag.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
print("Resultados salvos em resultados_parent_rag.json")